# CartPole policies and policy-variance diagnostics

This executable example accompanies the QASP release paper. It compares two decoded-Born policies on `CartPole-v1` using the same five-qubit graph-IQP circuit, trainable tensors, optimizer settings, and seeds. Only the readout changes: recursive parity uses an exact logical expectation, whereas majority-modulo uses an exact sparse action projector. The final section demonstrates `qasp.policy_variance` at a fixed observation and action.

The experiment is intentionally small and is an integration example, not a claim that one decoder is universally superior.

In [ ]:
from __future__ import annotations

import copy
import os
import time
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from qasp import REINFORCE, create_model, draw_model, policy_variance

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'qasp').is_dir():
    REPO_ROOT = REPO_ROOT.parent

PAPER_FIGURES = Path(
    os.environ.get('QASP_PAPER_FIGURES', REPO_ROOT / 'qasp_paper' / 'figures')
)
RESULTS_DIR = REPO_ROOT / 'examples' / 'results'
PAPER_FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_QUBITS = 5
N_ACTIONS = 2
N_LAYERS = 1
SEEDS = (3, 7, 11, 23, 37)
NUM_EPISODES = 300
MOVING_WINDOW = 20

print('PyTorch:', torch.__version__)
print('Gymnasium:', gym.__version__)
print('Paper figures:', PAPER_FIGURES)

## Matched QASP policy construction

For every seed, `build_matched_policies` constructs the two policies from a shared configuration and then copies the complete quantum-layer state from the parity model to the majority model. The assertions make the experimental control explicit.

In [ ]:
def build_matched_policies(seed: int):
    common = {
        'ansatz': 'IQPAnsatz',
        'ansatz_config': {
            'n_layers': N_LAYERS,
            'imprimitive_pattern': 'ring',
            'reuploading': True,
            'include_single_qubit': True,
        },
        'feature_map': 'NeuralEmbedding',
        'feature_map_config': {'features': 4, 'rotations': 'Z'},
        'n_actions': N_ACTIONS,
        'init': 'normal_0_1',
        'device': 'lightning.qubit',
        'diff_method': 'adjoint',
        'shots': None,
        'measurement_qubits': list(range(N_QUBITS)),
        'softmax_activation': False,
        'output_scaling': None,
        'torchrl': False,
        'parity_cnot_decomposition': False,
    }
    parity_config = {
        **copy.deepcopy(common),
        'policy': 'parity-like',
        'training_method': 'expectation',
    }
    majority_config = {
        **copy.deepcopy(common),
        'policy': 'majority-modulo',
        'training_method': 'sparse_projectors',
        'majority_blocks': [list(range(N_QUBITS))],
        'strict_majority_blocks': True,
        'enable_mpi': False,
    }

    torch.manual_seed(seed)
    np.random.seed(seed)
    parity = create_model('PQCBorn', parity_config, n_qubits=N_QUBITS)

    torch.manual_seed(seed)
    np.random.seed(seed)
    majority = create_model('PQCBorn', majority_config, n_qubits=N_QUBITS)
    majority.qlayer.load_state_dict(parity.qlayer.state_dict())

    parity_parameters = dict(parity.named_parameters())
    majority_parameters = dict(majority.named_parameters())
    assert parity_parameters.keys() == majority_parameters.keys()
    assert all(
        torch.equal(parity_parameters[name], majority_parameters[name])
        for name in parity_parameters
    )
    assert majority.majority_blocks == (tuple(range(N_QUBITS)),)
    return parity, majority


parity_example, majority_example = build_matched_policies(SEEDS[0])
zero_observation = torch.zeros(4)
print('Parity probabilities :', parity_example(zero_observation).detach())
print('Majority probabilities:', majority_example(zero_observation).detach())
print('Trainable parameters  :', sum(p.numel() for p in parity_example.parameters()))

In [ ]:
# QASP's drawing path resolves the same model configuration used for training.
circuit_figure, _ = draw_model(
    model=parity_example,
    policy_class='PQCBorn',
    n_qubits=N_QUBITS,
    num_inputs=4,
    inputs=zero_observation,
)
circuit_figure.suptitle('Five-qubit graph-IQP CartPole policy', y=1.02)
plt.show()

## Seeded CartPole executions

The trainer receives probabilities (`logits=False`) and uses identical REINFORCE hyperparameters for both readouts. Returns below are the unshaped episodic rewards reported by Gymnasium.

In [ ]:
def make_agent(policy, seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    agent = REINFORCE(
        policy=policy,
        env_name='CartPole-v1',
        gamma=0.99,
        lr=3e-2,
        batch_episodes=5,
        logits=False,
        normalize_mode='clip',
        normalize_returns=True,
        entropy_coef=5e-3,
        max_steps=200,
        baseline_type='ema',
        baseline_momentum=0.05,
        max_grad_norm=1.0,
    )
    agent.env.reset(seed=seed)
    agent.env.action_space.seed(seed)
    return agent


def run_cartpole():
    records = []
    timings = []
    for seed in SEEDS:
        parity, majority = build_matched_policies(seed)
        for policy_name, policy in (('recursive parity', parity), ('majority-modulo', majority)):
            agent = make_agent(policy, seed)
            started = time.perf_counter()
            returns = agent.train(num_episodes=NUM_EPISODES, log_every=100)
            elapsed = time.perf_counter() - started
            agent.env.close()
            timings.append({'seed': seed, 'policy': policy_name, 'seconds': elapsed})
            records.extend(
                {'seed': seed, 'policy': policy_name, 'episode': episode, 'return': reward}
                for episode, reward in enumerate(returns, start=1)
            )
            print(
                f'{policy_name:18s} seed={seed:2d}: '
                f'last-50 mean={np.mean(returns[-50:]):6.2f}, {elapsed:6.1f} s'
            )
    return pd.DataFrame(records), pd.DataFrame(timings)


returns_df, timings_df = run_cartpole()
returns_df.to_csv(RESULTS_DIR / 'cartpole_policy_returns.csv', index=False)
timings_df.to_csv(RESULTS_DIR / 'cartpole_policy_timings.csv', index=False)
timings_df

In [ ]:
def moving_average(values, window=MOVING_WINDOW):
    return np.convolve(values, np.ones(window) / window, mode='valid')


colors = {'recursive parity': '#7651c9', 'majority-modulo': '#e07a32'}
fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=180)
summary_rows = []
for policy_name, policy_frame in returns_df.groupby('policy', sort=False):
    runs = np.stack([
        frame.sort_values('episode')['return'].to_numpy()
        for _, frame in policy_frame.groupby('seed')
    ])
    smoothed = np.stack([moving_average(run) for run in runs])
    episodes = np.arange(MOVING_WINDOW, NUM_EPISODES + 1)
    mean = smoothed.mean(axis=0)
    sem = smoothed.std(axis=0, ddof=1) / np.sqrt(smoothed.shape[0])
    ax.plot(episodes, mean, color=colors[policy_name], lw=2.0, label=policy_name)
    ax.fill_between(episodes, mean - sem, mean + sem, color=colors[policy_name], alpha=0.18)
    final_means = runs[:, -50:].mean(axis=1)
    summary_rows.append({
        'policy': policy_name,
        'final-50 mean': final_means.mean(),
        'final-50 SEM': final_means.std(ddof=1) / np.sqrt(len(final_means)),
        'mean runtime (s)': timings_df.loc[timings_df.policy == policy_name, 'seconds'].mean(),
    })

ax.axhline(195, color='#444444', ls='--', lw=1.1, label='195 reference level')
ax.set(xlabel='Episode', ylabel=f'Return ({MOVING_WINDOW}-episode moving mean)', ylim=(0, 205))
ax.grid(alpha=0.22)
ax.legend(frameon=False, ncol=1)
fig.tight_layout()
fig.savefig(PAPER_FIGURES / 'cartpole_policy_rewards.pdf', bbox_inches='tight')
fig.savefig(PAPER_FIGURES / 'cartpole_policy_rewards.png', dpi=300, bbox_inches='tight')
plt.show()

cartpole_summary = pd.DataFrame(summary_rows)
cartpole_summary.to_csv(RESULTS_DIR / 'cartpole_policy_summary.csv', index=False)
cartpole_summary.round(2)

## `policy_variance` example

For a fixed state $s_0$ and action $a=0$, QASP estimates initialization variance

$$\widehat{\operatorname{Var}}[\pi_\Theta(0\mid s_0)] = \frac{1}{N}\sum_{j=1}^N (p_j-\bar p)^2,$$

where every $p_j$ is evaluated after an independent policy reinitialization. Fixing the action avoids the extra randomness introduced by sampling an action and then recording its probability. The example calls the public helper separately for every width so each majority block can be stated explicitly.

In [ ]:
VARIANCE_QUBITS = (3, 5, 7, 9)
VARIANCE_SAMPLES = 500
FIXED_STATE = torch.zeros(4)


def variance_config(n_qubits: int, policy_name: str):
    config = {
        'ansatz': 'IQPAnsatz',
        'ansatz_config': {
            'n_layers': 1,
            'imprimitive_pattern': 'ring',
            'reuploading': True,
            'include_single_qubit': True,
        },
        'feature_map': 'NeuralEmbedding',
        'feature_map_config': {'features': 4, 'rotations': 'Z'},
        'n_actions': 2,
        'init': 'normal_0_1',
        'device': 'lightning.qubit',
        'diff_method': 'adjoint',
        'shots': None,
        'measurement_qubits': list(range(n_qubits)),
        'softmax_activation': False,
        'output_scaling': None,
        'torchrl': False,
        'parity_cnot_decomposition': False,
    }
    if policy_name == 'recursive parity':
        config.update(policy='parity-like', training_method='expectation')
    else:
        config.update(
            policy='majority-modulo',
            training_method='sparse_projectors',
            majority_blocks=[list(range(n_qubits))],
            strict_majority_blocks=True,
            enable_mpi=False,
        )
    return config


variance_rows = []
for policy_name in ('recursive parity', 'majority-modulo'):
    for n_qubits in VARIANCE_QUBITS:
        variances, samples = policy_variance(
            policy_class='PQCBorn',
            policy_config_kwargs=variance_config(n_qubits, policy_name),
            action=0,
            qubit_list=[n_qubits],
            n_samples=VARIANCE_SAMPLES,
            batch_size=VARIANCE_SAMPLES,
            parallel=False,
            state=FIXED_STATE,
        )
        variance_rows.append({
            'policy': policy_name,
            'qubits': n_qubits,
            'samples': VARIANCE_SAMPLES,
            'variance': float(variances[0]),
        })

variance_df = pd.DataFrame(variance_rows)
variance_df.to_csv(RESULTS_DIR / 'policy_variance_fixed_state.csv', index=False)
variance_df

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=180)
for policy_name, frame in variance_df.groupby('policy', sort=False):
    ax.plot(
        frame['qubits'], frame['variance'], marker='o', ms=5.5, lw=2.0,
        color=colors[policy_name], label=policy_name,
    )
ax.set(
    xlabel='Number of qubits',
    ylabel=r'$\widehat{\mathrm{Var}}[\pi_\Theta(0\mid s_0)]$',
    xticks=list(VARIANCE_QUBITS),
    yscale='log',
)
ax.grid(alpha=0.22, which='both')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PAPER_FIGURES / 'policy_variance_example.pdf', bbox_inches='tight')
fig.savefig(PAPER_FIGURES / 'policy_variance_example.png', dpi=300, bbox_inches='tight')
plt.show()

## Reproduction notes

- The CSV files under `examples/results/` contain every raw episodic return, runtime, and variance estimate used in the figures.
- The CartPole plot shows the mean and standard error over five seeds after a 20-episode moving average.
- The 195 line is a visual reference inherited from common CartPole-v0 reporting; it is not used here as a statistical hypothesis test.
- `policy_variance(..., action=0, state=torch.zeros(4))` estimates parameter-initialization variance at one controlled input. Leaving `action=None` changes the estimand because an action is sampled before its probability is recorded.
- These small runs demonstrate the software path. Larger benchmark claims should increase seeds and episodes and report the full environment, library, and hardware configuration.